# Python Notebook Code for Pulse Signal Analysis

This notebook contains the Python workflow used for:

1. Data loading and preprocessing  
2. Exploratory Data Analysis (EDA)  
3. Statistical testing  
4. PyCaret-based machine learning modeling  

The dataset is expected to be available as:

```text
synthetic_pulse_data.csv
```

The target column used for classification is:

```text
heart_disease
```


## 1. Data Generation and Preprocessing

In [ ]:
# Import required libraries

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

# PyCaret is imported later inside a try-except block
# because it may not be installed in every notebook environment.

plt.rcParams["figure.figsize"] = (8, 6)
sns.set_theme(style="whitegrid")

print("Libraries imported successfully.")


In [ ]:
# Load dataset

DATA_PATH = "synthetic_pulse_data.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape of dataset:", df.shape)

df.head()


In [ ]:
# Basic information about the dataset

df.info()


In [ ]:
# Descriptive statistics

df.describe(include="all")


In [ ]:
# Check missing values

missing_values = df.isnull().sum().sort_values(ascending=False)

missing_values


In [ ]:
# Basic preprocessing

# Remove duplicate rows, if any
df = df.drop_duplicates()

# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

# Fill missing numeric values with median
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill missing categorical values with mode
for col in categorical_cols:
    if not df[col].mode().empty:
        df[col] = df[col].fillna(df[col].mode()[0])

print("Preprocessing completed.")
print("Updated shape:", df.shape)


## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Distribution of target variable

target_col = "heart_disease"

if target_col in df.columns:
    ax = sns.countplot(data=df, x=target_col)
    plt.title("Distribution of Heart Disease Classes")
    plt.xlabel("Heart Disease")
    plt.ylabel("Count")
    plt.show()
else:
    print(f"Target column '{target_col}' not found in dataset.")


In [ ]:
# Correlation heatmap for numeric columns only

corr_data = df.select_dtypes(include=[np.number]).corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_data, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap of Pulse Features")
plt.tight_layout()
plt.show()


In [ ]:
# Scatter plot example: Kapha BPM vs Vaata BPM

required_cols = ["Kapha_BPM", "Vaata_BPM", "heart_disease"]

if all(col in df.columns for col in required_cols):
    sns.scatterplot(
        data=df,
        x="Kapha_BPM",
        y="Vaata_BPM",
        hue="heart_disease"
    )
    plt.title("Kapha BPM vs Vaata BPM")
    plt.xlabel("Kapha BPM")
    plt.ylabel("Vaata BPM")
    plt.tight_layout()
    plt.show()
else:
    missing = [col for col in required_cols if col not in df.columns]
    print("Missing columns:", missing)


In [ ]:
# Pairplot for pulse BPM features

pulse_bpm_cols = [col for col in ["Kapha_BPM", "Pitta_BPM", "Vaata_BPM"] if col in df.columns]

if len(pulse_bpm_cols) >= 2 and target_col in df.columns:
    sns.pairplot(df[pulse_bpm_cols + [target_col]], hue=target_col, diag_kind="kde")
    plt.suptitle("Pairwise Relationship of Pulse BPM Features", y=1.02)
    plt.show()
else:
    print("Required pulse BPM columns or target column are not available.")


In [ ]:
# Boxplots for pulse features by heart disease class

pulse_features = [col for col in ["Kapha_BPM", "Pitta_BPM", "Vaata_BPM"] if col in df.columns]

if target_col in df.columns:
    for feature in pulse_features:
        sns.boxplot(data=df, x=target_col, y=feature)
        plt.title(f"{feature} Distribution by Heart Disease Class")
        plt.xlabel("Heart Disease")
        plt.ylabel(feature)
        plt.tight_layout()
        plt.show()
else:
    print(f"Target column '{target_col}' not found.")


## 3. Statistical Tests

The following section compares pulse features between the heart disease and non-heart disease groups.  

For each feature:

- Shapiro-Wilk test checks approximate normality.
- Independent t-test is used when both groups are approximately normal.
- Mann-Whitney U test is used when normality is not satisfied.

A p-value below 0.05 is treated as statistically significant.


In [ ]:
# Statistical testing between target classes

def compare_groups(data, feature, target):
    classes = sorted(data[target].dropna().unique())

    if len(classes) != 2:
        return {
            "Feature": feature,
            "Test Used": "Not applicable",
            "Reason": "Target column must contain exactly two classes.",
            "Statistic": np.nan,
            "p-value": np.nan,
            "Significant at 0.05": "No"
        }

    group_1 = data[data[target] == classes[0]][feature].dropna()
    group_2 = data[data[target] == classes[1]][feature].dropna()

    # Shapiro-Wilk normality test is suitable for small to moderate samples.
    # For very large samples, it can become overly sensitive.
    shapiro_1_p = stats.shapiro(group_1.sample(min(len(group_1), 500), random_state=123))[1]
    shapiro_2_p = stats.shapiro(group_2.sample(min(len(group_2), 500), random_state=123))[1]

    normal = shapiro_1_p > 0.05 and shapiro_2_p > 0.05

    if normal:
        test_name = "Independent t-test"
        statistic, p_value = stats.ttest_ind(group_1, group_2, equal_var=False)
    else:
        test_name = "Mann-Whitney U test"
        statistic, p_value = stats.mannwhitneyu(group_1, group_2, alternative="two-sided")

    return {
        "Feature": feature,
        "Class 1": classes[0],
        "Class 2": classes[1],
        "Shapiro p-value class 1": shapiro_1_p,
        "Shapiro p-value class 2": shapiro_2_p,
        "Test Used": test_name,
        "Statistic": statistic,
        "p-value": p_value,
        "Significant at 0.05": "Yes" if p_value < 0.05 else "No"
    }


if target_col in df.columns:
    test_results = []

    for feature in pulse_features:
        test_results.append(compare_groups(df, feature, target_col))

    statistical_results = pd.DataFrame(test_results)
    statistical_results
else:
    print(f"Target column '{target_col}' not found.")


## 4. PyCaret Modeling

This section builds classification models using PyCaret.  
The notebook compares multiple baseline models, finalizes the best model, and saves it as:

```text
ppg_cvd_model.pkl
```


In [ ]:
# PyCaret modeling

try:
    from pycaret.classification import setup, compare_models, finalize_model, save_model, pull

    clf = setup(
        data=df,
        target="heart_disease",
        session_id=123,
        verbose=False
    )

    best_model = compare_models()

    comparison_results = pull()

    print("Best model selected:")
    print(best_model)

    comparison_results

except ModuleNotFoundError:
    print("PyCaret is not installed in this environment.")
    print("Install it using:")
    print("pip install pycaret")
except Exception as e:
    print("PyCaret modeling failed.")
    print("Error:", e)


In [ ]:
# Finalize and save the best model

try:
    final_model = finalize_model(best_model)

    save_model(final_model, "ppg_cvd_model")

    print("Final model saved successfully as ppg_cvd_model.pkl")

except NameError:
    print("best_model is not available. Run the PyCaret modeling cell first.")
except Exception as e:
    print("Model finalization or saving failed.")
    print("Error:", e)


## 5. Notes for Reproducibility

- Keep the CSV file in the same folder as this notebook.
- Ensure that the target column is named `heart_disease`.
- Ensure that the pulse feature columns are named consistently, such as:
  - `Kapha_BPM`
  - `Pitta_BPM`
  - `Vaata_BPM`
- PyCaret must be installed before running the modeling cells.
